# AWS Cloud APIs 

https://boto3.amazonaws.com/v1/documentation/api/latest/index.html

## Credential safety

The named profile and credential-inspection cells below illustrate how the SDK resolves a local session; replace the example profile with your dedicated course profile. In normal application code, use the default credential provider chain or a temporary role and never read, print, log, or save access keys in notebook variables. Treat any exposed credential as compromised, rotate it immediately, and verify that notebooks contain no secrets before committing them.

In [1]:
#!pip install boto3

In [2]:
import os
import boto3

In [3]:
region_name = 'us-east-1'

In [5]:
# Create a session using a specific profile
session = boto3.Session(profile_name='default')

In [6]:
# OPTIONAL DEBUG STEP

# Access the credentials
credentials = session.get_credentials()

# Access the access key ID and secret access key
aws_access_key_id = credentials.access_key
aws_secret_access_key = credentials.secret_key

# aws_secret_access_key, aws_access_key_id

In [7]:
def get_session_info():
    """Get basic information about the AWS session."""
    session = boto3.session.Session()
    
    # Get current region
    current_region = session.region_name
    print(f"Current region: {current_region or 'Not set'}")
    
    # Get available regions
    ec2 = boto3.client('ec2', region_name=current_region or 'us-east-1')
    regions = [region['RegionName'] for region in ec2.describe_regions()['Regions']]
    print(f"Available regions: {', '.join(regions)}")
    
    # Get account info
    sts = boto3.client('sts')
    account_id = sts.get_caller_identity()['Account']
    print(f"AWS Account ID: {account_id}")
    
    # Get identity info
    user = sts.get_caller_identity()['Arn']
    print(f"IAM Identity: {user}")
    
    # Get available services
    services = session.get_available_services()
    print(f"Available AWS services: {len(services)}")
    print(f"Sample services: {', '.join(sorted(services)[:10])}")

In [8]:
get_session_info()

Current region: us-east-1
Available regions: ap-south-1, eu-north-1, eu-west-3, eu-west-2, eu-west-1, ap-northeast-3, ap-northeast-2, ap-northeast-1, ca-central-1, sa-east-1, ap-southeast-1, ap-southeast-2, eu-central-1, us-east-1, us-east-2, us-west-1, us-west-2
AWS Account ID: 673868928338
IAM Identity: arn:aws:iam::673868928338:user/ashish
Available AWS services: 436
Sample services: accessanalyzer, account, account-access, acm, acm-pca, agent-registry, agent-registry-control, aiops, amp, amplify


## S3 - API

https://boto3.amazonaws.com/v1/documentation/api/1.20.5/reference/services/s3.html

In [9]:
# Access the configured S3 client. 
# The region_name is optional; if not specified, Boto3 will use the default region ('us-east-1' for AWS).
s3_client = session.client('s3', region_name='us-east-1')

In [10]:
def list_files(bucket_name, prefix=''):
    files = []
    paginator = s3_client.get_paginator('list_objects_v2')
    page_iterator = paginator.paginate(Bucket=bucket_name, Prefix=prefix)
    
    for page in page_iterator:
        if 'Contents' in page:
            for obj in page['Contents']:
                files.append(obj['Key'])
        
    return files

def create_bucket(bucket_name):
    s3_client.create_bucket(
        Bucket=bucket_name
    )
    
def download_file(file_key, local_file_path):
    s3_client.download_file(bucket_name, file_key, local_file_path)

def upload_file(local_file_path, file_key):
    s3_client.upload_file(local_file_path, bucket_name, file_key)

In [11]:
# List all buckets:
response = s3_client.list_buckets()
for bucket in response['Buckets']:
    print(f'Bucket Name: {bucket["Name"]}')

Bucket Name: athena-query-results-xxxxxx
Bucket Name: bigdata-ashish
Bucket Name: bigdata-ashish1
Bucket Name: cloud-engineering-01723013101230apujari
Bucket Name: cloud-engineering-apujari
Bucket Name: cloud-engineeringapujari
Bucket Name: elasticbeanstalk-us-east-1-673868928338
Bucket Name: witbit-ai


In [12]:
# Bucket name - ensure it is unique and no uppercase 
bucket_name = 'cloud-engineering'+os.getlogin()

In [13]:
create_bucket(bucket_name)

In [14]:
# List all buckets:
response = s3_client.list_buckets()
for bucket in response['Buckets']:
    print(f'Bucket Name: {bucket["Name"]}')

Bucket Name: athena-query-results-xxxxxx
Bucket Name: bigdata-ashish
Bucket Name: bigdata-ashish1
Bucket Name: cloud-engineering-01723013101230apujari
Bucket Name: cloud-engineering-apujari
Bucket Name: cloud-engineeringapujari
Bucket Name: elasticbeanstalk-us-east-1-673868928338
Bucket Name: witbit-ai


In [15]:
# List of files
print("List of files:")
list_files(bucket_name)

List of files:


['Country-data.csv']

In [16]:
# file key
file_key = 'Country-data.csv'
local_file_path = '../Country-data.csv'

print("\nUploading file:", local_file_path, "to key:", file_key)
upload_file(local_file_path, file_key)


Uploading file: ../Country-data.csv to key: Country-data.csv


In [17]:
print (bucket_name)

cloud-engineeringapujari


In [18]:
print("List of files after upload:")
list_files(bucket_name)

List of files after upload:


['Country-data.csv']

In [19]:
print("Downloading file:", file_key)
download_file(file_key, local_file_path+"-downloaded")

In [20]:
# List of files
print("List of files:")
list_files(bucket_name)

List of files:


['Country-data.csv']

## EC2 - API

https://boto3.amazonaws.com/v1/documentation/api/1.20.5/reference/services/ec2.html

In [21]:
# Create EC2 client 
ec2_client = boto3.client('ec2',region_name=region_name)

In [22]:
ec2_client.describe_regions()

{'Regions': [{'OptInStatus': 'opt-in-not-required',
   'Geography': [{'Name': 'India'}],
   'RegionName': 'ap-south-1',
   'Endpoint': 'ec2.ap-south-1.amazonaws.com'},
  {'OptInStatus': 'opt-in-not-required',
   'Geography': [{'Name': 'Sweden'}],
   'RegionName': 'eu-north-1',
   'Endpoint': 'ec2.eu-north-1.amazonaws.com'},
  {'OptInStatus': 'opt-in-not-required',
   'Geography': [{'Name': 'France'}],
   'RegionName': 'eu-west-3',
   'Endpoint': 'ec2.eu-west-3.amazonaws.com'},
  {'OptInStatus': 'opt-in-not-required',
   'Geography': [{'Name': 'United Kingdom'}],
   'RegionName': 'eu-west-2',
   'Endpoint': 'ec2.eu-west-2.amazonaws.com'},
  {'OptInStatus': 'opt-in-not-required',
   'Geography': [{'Name': 'Ireland'}],
   'RegionName': 'eu-west-1',
   'Endpoint': 'ec2.eu-west-1.amazonaws.com'},
  {'OptInStatus': 'opt-in-not-required',
   'Geography': [{'Name': 'Japan'}],
   'RegionName': 'ap-northeast-3',
   'Endpoint': 'ec2.ap-northeast-3.amazonaws.com'},
  {'OptInStatus': 'opt-in-not-re

In [23]:
ec2_client.describe_vpcs()

{'Vpcs': [{'OwnerId': '673868928338',
   'InstanceTenancy': 'default',
   'CidrBlockAssociationSet': [{'AssociationId': 'vpc-cidr-assoc-03277d1656305ce71',
     'CidrBlock': '172.31.0.0/16',
     'CidrBlockState': {'State': 'associated'}}],
   'IsDefault': True,
   'BlockPublicAccessStates': {'InternetGatewayBlockMode': 'off'},
   'VpcId': 'vpc-0fe450a9cd4623c54',
   'State': 'available',
   'CidrBlock': '172.31.0.0/16',
   'DhcpOptionsId': 'dopt-04f25dc3a2e62b49e'}],
 'ResponseMetadata': {'RequestId': '6673c3b0-fc59-4971-9d91-e2f5af41a66e',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': '6673c3b0-fc59-4971-9d91-e2f5af41a66e',
   'cache-control': 'no-cache, no-store',
   'strict-transport-security': 'max-age=31536000; includeSubDomains',
   'content-type': 'text/xml;charset=UTF-8',
   'content-length': '799',
   'date': 'Mon, 05 Oct 2026 18:36:13 GMT',
   'server': 'AmazonEC2'},
  'RetryAttempts': 0}}

In [24]:
ec2_client.describe_instances()

{'Reservations': [{'ReservationId': 'r-02d81de9202b06a0e',
   'OwnerId': '673868928338',
   'Groups': [],
   'Instances': [{'Architecture': 'x86_64',
     'BlockDeviceMappings': [],
     'ClientToken': 'bf84af75-b4c2-4342-a579-ee818edf568e',
     'EbsOptimized': True,
     'EnaSupport': True,
     'Hypervisor': 'xen',
     'NetworkInterfaces': [],
     'RootDeviceName': '/dev/sda1',
     'RootDeviceType': 'ebs',
     'SecurityGroups': [],
     'StateReason': {'Code': 'Client.UserInitiatedShutdown',
      'Message': 'Client.UserInitiatedShutdown: User initiated shutdown'},
     'Tags': [{'Key': 'Name', 'Value': 'postgres dev'}],
     'VirtualizationType': 'hvm',
     'CpuOptions': {'CoreCount': 1, 'ThreadsPerCore': 2},
     'CapacityReservationSpecification': {'CapacityReservationPreference': 'open'},
     'HibernationOptions': {'Configured': False},
     'MetadataOptions': {'State': 'pending',
      'HttpTokens': 'required',
      'HttpPutResponseHopLimit': 2,
      'HttpEndpoint': 'en